# Note on R² vs Sharpe Ratio

**Question.** Last month's return explains about 1% of this month's return: a regression of this month's return on last month's has an $R^2$ of 0.01. That sounds useless. Is it worth trading on?

**Answer.** Yes, more than you might think. If you trade on the forecast, the strategy's Sharpe ratio per period is

$$SR = \sqrt{\frac{R^2}{1+R^2}} \;\approx\; \sqrt{R^2}.$$

With monthly data and $R^2 = 0.01$, that is $0.10$ a month, or $0.10 \times \sqrt{12} \approx 0.35$ a year: about the Sharpe ratio of the stock market itself.

Part 1 derives the formula step by step. Part 2 checks it by simulation. Part 3 has short exercises that bend the assumptions. Part 4 asks whether 50 years of data are enough to tell such a strategy from luck.

*Tip: if a step is unclear, ask the Gemini panel in Colab to explain it.*

> **A note on how this notebook was made.** It was written by an AI (Claude, from Anthropic), working from your instructor's derivation. Treat it the way you would treat any code you did not write: run it, read it, and check its claims against the output you see. If something looks wrong, tell your instructor.

## Part 1. The derivation

**Setup.** Measure returns relative to their average, so everything has mean zero. Regress next period's return on the predictor (here, the past return) and split the return into two pieces:

$$r = s + \varepsilon$$

- $s$ is the **forecast** (the fitted value), with variance $\sigma_s^2$.
- $\varepsilon$ is the **forecast error**, with variance $\sigma_\varepsilon^2$. It is unrelated to $s$.

**Step 1. Write $R^2$ in terms of these pieces.** $R^2$ is the share of the variance the forecast explains:

$$R^2 = \frac{\sigma_s^2}{\sigma_s^2 + \sigma_\varepsilon^2}
\quad\Longrightarrow\quad
\sigma_\varepsilon^2 = \sigma_s^2 \, \frac{1-R^2}{R^2}.$$

**Step 2. Define the strategy.** Hold a position equal to the forecast: long when $s>0$, short when $s<0$. The strategy's return is position times return:

$$\pi = s \cdot r = s\,(s + \varepsilon) = s^2 + s\,\varepsilon.$$

**Step 3. Expected return.** The second term averages to zero because the error is unrelated to the forecast:

$$E[\pi] = E[s^2] + E[s\,\varepsilon] = \sigma_s^2 + 0 = \sigma_s^2.$$

**Step 4. Variance.** Again there is no covariance term, because $\varepsilon$ is unrelated to $s$:

$$\operatorname{Var}(\pi) = \operatorname{Var}(s^2) + \operatorname{Var}(s\,\varepsilon).$$

- $\operatorname{Var}(s^2) = E[s^4] - (E[s^2])^2 = 3\sigma_s^4 - \sigma_s^4 = 2\sigma_s^4$. For a normal variable $E[s^4] = 3\sigma_s^4$ (kurtosis 3). Note this is *not* $\sigma_s^2$: $s^2$ is in units of return², so its variance is in return⁴. This term is the risk from the *size* of the bet changing over time.
- $\operatorname{Var}(s\,\varepsilon) = \sigma_s^2\,\sigma_\varepsilon^2$.

Substitute $\sigma_\varepsilon^2$ from Step 1:

$$\operatorname{Var}(\pi) = 2\sigma_s^4 + \sigma_s^4\,\frac{1-R^2}{R^2} = \sigma_s^4\,\frac{1+R^2}{R^2}.$$

**Step 5. Sharpe ratio = mean ÷ standard deviation.** $\sigma_s^2$ cancels:

$$SR = \frac{\sigma_s^2}{\sqrt{\sigma_s^4\,(1+R^2)/R^2}} = \sqrt{\frac{R^2}{1+R^2}}.$$

**Step 6. Simplify for small $R^2$.** When $R^2$ is small, $1+R^2 \approx 1$, so $SR \approx \sqrt{R^2}$ per period. $\sqrt{R^2}$ is the correlation between forecast and return.

**Step 7. Annualize.** Multiply by $\sqrt{\text{periods per year}}$. Monthly, with $R^2 = 0.01$: $SR \approx 0.10 \times \sqrt{12} \approx 0.35$ a year.

**Assumptions used:** (1) returns are measured relative to their mean; (2) the forecast error is unrelated to the forecast; (3) returns are normal (used only for $E[s^4] = 3\sigma_s^4$); (4) the $R^2$ holds out of sample. Notice that the size of the position does not matter: doubling it doubles both the mean and the standard deviation.

## Part 2. Check it by simulation

We simulate monthly returns where this month's return depends on last month's:

$$r_t = \rho\, r_{t-1} + e_t.$$

The forecast is $s_t = \rho\, r_{t-1}$, so $R^2 = \rho^2$. The strategy holds $s_t$ and earns $s_t \cdot r_t$. Monthly volatility is 4.5% (about 16% a year, like the stock market).

In [ ]:
import numpy as np, pandas as pd
import matplotlib.pyplot as plt
from scipy.signal import lfilter

In [ ]:
pd.set_option("display.float_format", "{:,.3f}".format)

SIGMA = 0.045                    # monthly return volatility
rng = np.random.default_rng(736)

def simulate(r2, T, dist="normal", df=6, position="forecast", cost_bp=0.0, sigma=SIGMA):
    '''Simulate T periods of r_t = rho*r_{t-1} + e_t with R^2 = rho^2, trade on the forecast,
    and return the strategy's Sharpe ratio per period.

    dist:     "normal", or "t" for fat-tailed shocks (Student t with `df` degrees of freedom)
    position: "forecast" holds s_t (scaled to be about 1 on average); "sign" holds +1 or -1
    cost_bp:  trading cost in basis points per unit of position traded
    '''
    rho = np.sqrt(r2)
    if dist == "normal":
        e = rng.standard_normal(T + 1)
    else:
        e = rng.standard_t(df, T + 1) / np.sqrt(df / (df - 2))   # unit variance
    e *= sigma * np.sqrt(1 - r2)                        # keeps var(r) = sigma^2
    e[0] /= np.sqrt(1 - r2)                             # start r_0 at its long-run variance
    r = lfilter([1.0], [1.0, -rho], e)                  # r_t = rho*r_{t-1} + e_t, without a slow loop
    s = rho * r[:-1]                                    # forecast of r[1:], made one period earlier
    w = s / (rho * sigma) if position == "forecast" else np.sign(s)
    pnl = w * r[1:] - cost_bp / 1e4 * np.abs(np.diff(w, prepend=0.0))
    return pnl.mean() / pnl.std()

def formula(r2):
    return np.sqrt(r2 / (1 + r2))

In [ ]:
T_LONG = 2_000_000                # a very long sample, so the simulated Sharpe ratio is precise
r2_grid = [0.001, 0.005, 0.01, 0.02, 0.05, 0.10, 0.20]

table = pd.DataFrame({
    "R2": r2_grid,
    "sqrt(R2)": np.sqrt(r2_grid),
    "formula: sqrt(R2/(1+R2))": [formula(x) for x in r2_grid],
    "simulated": [simulate(x, T_LONG) for x in r2_grid],
})
table["annual (x sqrt 12)"] = table["simulated"] * np.sqrt(12)
table

In [ ]:
grid = np.linspace(0.0005, 0.25, 200)
fig, ax = plt.subplots(figsize=(7, 3.5))
ax.plot(grid, formula(grid), label="formula: $\\sqrt{R^2/(1+R^2)}$")
ax.plot(grid, np.sqrt(grid), ls="--", label="approximation: $\\sqrt{R^2}$")
ax.scatter(table["R2"], table["simulated"], color="black", zorder=3, label="simulated")
ax.set_xlabel("$R^2$ of the forecast"); ax.set_ylabel("Sharpe ratio per period")
ax.set_title("A small $R^2$ still gives a sizeable Sharpe ratio"); ax.legend()
plt.tight_layout(); plt.show()

**Read it.** Do the simulated Sharpe ratios sit on the formula? Up to what $R^2$ is the approximation $\sqrt{R^2}$ good enough? Look at the steep part of the curve near zero: going from $R^2 = 0$ to $0.01$ adds as much Sharpe ratio as going from $0.01$ to $0.04$.

## Part 3. Bend the assumptions

Each exercise changes one argument of `simulate`. Predict the answer first, then run the cell.

### 3.1 Fat tails

Real returns have fatter tails than the normal. In Step 4 we used $E[s^4] = 3\sigma_s^4$. With kurtosis $\kappa$ instead of 3, the same algebra gives

$$SR = \sqrt{\frac{R^2}{1 + (\kappa - 2)\,R^2}}.$$

The cell below uses Student-$t$ shocks with 6 degrees of freedom (kurtosis 6). What matters in the formula is the kurtosis of the *forecast*, which is last month's return. That return adds up many past shocks, which thins its tails a little: its kurtosis is $3 + 3\,(1-R^2)/(1+R^2)$, just under 6 when $R^2$ is small.

In [ ]:
def formula_kurt(r2, kappa):
    return np.sqrt(r2 / (1 + (kappa - 2) * r2))

rows = []
for x in [0.01, 0.10, 0.20]:
    kappa = 3 + 3 * (1 - x) / (1 + x)                   # kurtosis of last month's return with t(6) shocks
    rows.append({"R2": x, "normal": simulate(x, T_LONG), "fat tails (t, df=6)": simulate(x, T_LONG, dist="t", df=6),
                 "kurtosis of forecast": kappa, "formula with that kurtosis": formula_kurt(x, kappa)})
pd.DataFrame(rows)

**Read it.** How much do fat tails cost at $R^2 = 0.01$? At $R^2 = 0.20$? Use the formula to explain why the answer depends on $R^2$. (Hint: which term in the denominator do fat tails change, and how big is it when $R^2$ is small?)

### 3.2 Trade only the sign

Instead of holding a position proportional to the forecast, hold $+1$ when the forecast is positive and $-1$ when it is negative. For small $R^2$, theory says you keep about $\sqrt{2/\pi} \approx 0.80$ of the Sharpe ratio. (Try deriving it: the mean becomes $E|s| = \sigma_s\sqrt{2/\pi}$.)

In [ ]:
rows = []
for x in [0.01, 0.05, 0.20]:
    full, sign = simulate(x, T_LONG), simulate(x, T_LONG, position="sign")
    rows.append({"R2": x, "forecast-weighted": full, "sign only": sign, "ratio": sign / full})
pd.DataFrame(rows)

**Read it.** Is the ratio close to 0.80 when $R^2$ is small? Why does it rise toward 1 as $R^2$ grows? (Hint: Step 4. Sizing the bet adds the $\operatorname{Var}(s^2)$ risk term; trading the sign does not.) Why does sizing the bet by the forecast help? When might sign-only trading be preferable in practice?

### 3.3 Trading costs and trading more often

Step 7 says more frequent trading raises the annual Sharpe ratio by $\sqrt{\text{periods per year}}$. But the forecast changes almost completely every period, so the strategy trades a lot.

The cell compares a monthly signal ($R^2 = 0.01$, 4.5% monthly volatility) with a daily signal that has the *same annual Sharpe ratio before costs* ($R^2 = 0.01 \times 12/252$, 1% daily volatility), at several trading costs.

In [ ]:
setups = {"monthly": dict(r2=0.01, sigma=0.045, per_year=12),
          "daily":   dict(r2=0.01 * 12 / 252, sigma=0.01, per_year=252)}

rows = []
for cost in [0, 5, 10, 20]:
    row = {"cost (bp per unit traded)": cost}
    for name, p in setups.items():
        sr = simulate(p["r2"], T_LONG, cost_bp=cost, sigma=p["sigma"])
        row[f"{name}: annual SR"] = sr * np.sqrt(p["per_year"])
    rows.append(row)
pd.DataFrame(rows).set_index("cost (bp per unit traded)")

**Read it.** Before costs, both signals have the same annual Sharpe ratio. Which one survives 10 bp of trading costs? Explain using the size of the forecast each period ($\sigma_s = \sqrt{R^2}\,\sigma$) relative to the cost of changing the position.

## Part 4. Can 50 years of data tell this from luck?

An annual Sharpe ratio of 0.35 is valuable. But the standard error of an estimated Sharpe ratio is about $1/\sqrt{T}$ per period, so the $t$-statistic is about $SR_{\text{annual}} \times \sqrt{\text{years}}$. The cell simulates 2,000 histories of 50 years each for a strategy with true $R^2 = 0.01$.

In [ ]:
YEARS, N_HIST = 50, 2000
sr_hat = np.array([simulate(0.01, 12 * YEARS) for _ in range(N_HIST)]) * np.sqrt(12)

print(f"True annual SR:          {formula(0.01) * np.sqrt(12):.2f}")
print(f"Mean of estimates:       {sr_hat.mean():.2f}")
print(f"Std of estimates:        {sr_hat.std():.2f}   (theory: 1/sqrt(years) = {1 / np.sqrt(YEARS):.2f})")
print(f"Share with t-stat < 2:   {(sr_hat * np.sqrt(YEARS) < 2).mean():.0%}")

fig, ax = plt.subplots(figsize=(7, 3))
ax.hist(sr_hat, bins=50, color="#1565c0", alpha=0.8)
ax.axvline(formula(0.01) * np.sqrt(12), color="black", lw=1.5, label="true SR")
ax.axvline(2 / np.sqrt(YEARS), color="grey", ls="--", label="t = 2 cutoff")
ax.set_xlabel("estimated annual Sharpe ratio, 50 years of monthly data"); ax.legend()
plt.tight_layout(); plt.show()

**Read it.** In what share of 50-year histories would a researcher fail to find this genuinely profitable strategy at $t > 2$? Now flip it around: among many useless predictors, some will show an $R^2$ of 0.01 by chance. McQueen and Thorley (1999, *Mining Fool's Gold*) put it this way: search through 100 variables and about 5 will be significant at the 95% level by chance alone. Their Foolish Four had a $t$-statistic of 3.05 over 24 years, yet beat the Dow by only 0.32 percentage points a year out of sample (1949–72). Which of their warning signs of data mining would you check before trading a predictor with $R^2 = 0.01$?

## Part 5. Connection to the fundamental law

In Week 1 we met Grinold's fundamental law of active management:

$$IR = IC \times \sqrt{\text{breadth}}.$$

This note is a special case. The information coefficient $IC$ is the correlation between forecast and return, which is $\sqrt{R^2}$. Breadth is the number of independent bets per year: 12 for a monthly signal on one asset. So

$$IR \approx \sqrt{R^2} \times \sqrt{12}.$$

Part 3.3 is the catch the formula leaves out: breadth bought by trading more often has to pay for itself after costs.